# DIMER Language-Model Artifact Inference
**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification `1.0`

This notebook consumes an **externally supplied PEFT adapter ZIP**. No training or fine-tuning occurs. It validates the artifact, reconstructs it through the production inference surface, accepts new input, and exports predictions. A successful run does **not** establish sender authenticity, task quality, safety, fairness, calibration, robustness, or production fitness.

[Pipeline repository](https://github.com/kurtvalcorza/language-model-pipeline) · [SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) · [PEFT](https://huggingface.co/docs/peft)


## Prerequisites and trust boundary
Use Google Colab with CUDA. **Private production source:** `language-model-finetuner` requires a `GITHUB_TOKEN` Colab Secret with **read access** to that private repository. The token is passed through an ephemeral Git header, never printed or embedded in the clone URL, and deleted from notebook state after checkout.

Upload exactly one external adapter ZIP. Manifest consistency is not sender authenticity. A whole-ZIP SHA-256 helps only if obtained through an independently trusted channel. Path-safe extraction is not permission to deserialize arbitrary model state; this path requires safetensors/JSON and `trustRemoteCode=false`.


In [ ]:
%pip -q install transformers==5.16.1 tokenizers==0.23.2 huggingface-hub==1.30.0 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.49.0 safetensors==0.8.0 datasets==4.8.5 pandas==2.3.3 PyYAML==6.0.3 Jinja2==3.1.6
%pip -q install --no-deps git+https://github.com/kurtvalcorza/language-model-pipeline.git@afaf1f032cd7e9751db1ee8eb71b542f9bd0b15f


In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
import torch
from lmpipeline.tutorial_runtime import checkout_private_finetuner, github_token_from_runtime, consume_adapter_archive
from lmpipeline.tutorial_api import assert_finetuner_checkout, assert_runtime_compatible, assert_tutorial_runtime, resolve_artifact_model, sha256_file, validate_prompt

PIPELINE_RUNTIME_REVISION = "afaf1f032cd7e9751db1ee8eb71b542f9bd0b15f"
FINETUNER_RUNTIME_REVISION = "3772f0ca4e0f7130ffd5f826ea40f43b7212339e"
FINETUNER_ROOT = Path("/content/language-model-finetuner")
_GITHUB_TOKEN = github_token_from_runtime()
checkout_private_finetuner(FINETUNER_ROOT, revision=FINETUNER_RUNTIME_REVISION, token=_GITHUB_TOKEN)
del _GITHUB_TOKEN
sys.path.insert(0, str(FINETUNER_ROOT / "src"))

from finetuner.inference import generate_reply, load_adapter_for_inference, verify_adapter_active
RUNTIME = assert_tutorial_runtime()
assert_finetuner_checkout(FINETUNER_ROOT, FINETUNER_RUNTIME_REVISION)
if not torch.cuda.is_available():
    raise RuntimeError("Artifact inference requires CUDA; select a Colab GPU runtime.")
print(json.dumps({"pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION, "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION, "runtime": RUNTIME}, indent=2))


## 1. Validate the external artifact
`consume_adapter_archive` enforces root-level manifest placement, safe paths, no symlinks/duplicates, an expansion limit, hashes/sizes, required files, and no unlisted files. The manifest must explicitly identify `format = peft_adapter` and `formatVersion = 1`.


In [ ]:
EXPECTED_ARTIFACT_ZIP_SHA256 = "" # @param {type:"string"}
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one adapter ZIP")
name, payload = next(iter(uploaded.items()))
if not name.lower().endswith(".zip"):
    raise ValueError("Artifact must be a .zip")
ARCHIVE = Path("/content") / Path(name).name
ARCHIVE.write_bytes(payload)
ARTIFACT_ROOT, MANIFEST, PROVENANCE = consume_adapter_archive(
    ARCHIVE, extraction_root="/content/dimer-language-model-artifact",
    expected_archive_sha256=EXPECTED_ARTIFACT_ZIP_SHA256,
)
if (MANIFEST.get("format"), MANIFEST.get("formatVersion")) != ("peft_adapter", 1):
    raise ValueError("Unsupported artifact manifest format/version")
print({"sha256": sha256_file(ARCHIVE), "format": MANIFEST["format"], "formatVersion": MANIFEST["formatVersion"], "files": len(MANIFEST["files"])})


## 2. Verify provenance and compatibility
The artifact must match the canonical base-model ID/revision and critical producer package versions. Tutorial artifacts also carry runtime-source revisions; those are enforced when present. Ordinary production artifacts may omit the tutorial-only `runtimeRevisions` extension, in which case canonical model/revision plus package compatibility remains authoritative.


In [ ]:
ENTRY = resolve_artifact_model(PROVENANCE)
assert_runtime_compatible(PROVENANCE, RUNTIME)
RECORDED_RUNTIME_REVISIONS = PROVENANCE.get("runtimeRevisions") or {}
if RECORDED_RUNTIME_REVISIONS:
    if set(RECORDED_RUNTIME_REVISIONS) != {"pipeline", "finetuner"}:
        raise ValueError("Partial runtimeRevisions provenance is not accepted")
    if RECORDED_RUNTIME_REVISIONS["pipeline"] != PIPELINE_RUNTIME_REVISION:
        raise ValueError("Pipeline runtime revision mismatch")
    if RECORDED_RUNTIME_REVISIONS["finetuner"] != FINETUNER_RUNTIME_REVISION:
        raise ValueError("Finetuner runtime revision mismatch")
    SOURCE_COMPATIBILITY = "tutorial runtime-source revisions matched"
else:
    SOURCE_COMPATIBILITY = "production artifact: canonical model/revision and package compatibility enforced"
print(json.dumps({"modelKey": ENTRY.key, "baseModel": ENTRY.model_id, "baseModelRevision": ENTRY.revision, "packageVersions": PROVENANCE.get("packageVersions"), "runtimeRevisions": RECORDED_RUNTIME_REVISIONS or None, "sourceCompatibility": SOURCE_COMPATIBILITY}, indent=2))


## 3. Reconstruct and prove the adapter is active
The exact base revision and serialized PEFT adapter are loaded through `finetuner.inference`. QLoRA uses the production 4-bit path. Non-quantized CUDA artifacts use BF16 when supported, otherwise FP16, avoiding accidental FP32 widening. Activity requires non-zero LoRA B weights and a non-zero adapter-on/off logit delta.


In [ ]:
QUANTIZED = bool(PROVENANCE.get("quantized"))
BASE_DTYPE = None if QUANTIZED else (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
MODEL, TOKENIZER = load_adapter_for_inference(
    ARTIFACT_ROOT, entry=ENTRY, device="cuda", quantized=QUANTIZED, base_dtype=BASE_DTYPE,
)
ACTIVITY = verify_adapter_active(MODEL, TOKENIZER, prompt="Kumusta.")
print({"adapterActivity": ACTIVITY, "quantized": QUANTIZED, "baseDtype": str(BASE_DTYPE)})


## 4. Run real new-data inference and export results
Edit `CUSTOM_PROMPT`. Greedy decoding is the deterministic verification path; stochastic temperature/top-p/top-k settings should be explicit when used. JSONL/CSV preserve input identity, prompt, output, model identity, and decoding. Prompt/output text may be sensitive.


In [ ]:
CUSTOM_PROMPT = "Sumulat ng dalawang pangungusap tungkol sa responsableng paggamit ng AI." # @param {type:"string"}
MAX_NEW_TOKENS = 128 # @param {type:"integer"}
TRAINING = (PROVENANCE.get("job") or {}).get("training") or {}
MAX_SEQUENCE_LENGTH = int(TRAINING.get("maxSequenceLength", 0))
if MAX_SEQUENCE_LENGTH <= 0 or MAX_SEQUENCE_LENGTH > int(ENTRY.max_sequence_length):
    raise ValueError("Invalid maxSequenceLength provenance")
PROMPT_TOKENS = validate_prompt(TOKENIZER, CUSTOM_PROMPT, max_sequence_length=MAX_SEQUENCE_LENGTH)
if MAX_NEW_TOKENS <= 0 or PROMPT_TOKENS + MAX_NEW_TOKENS > MAX_SEQUENCE_LENGTH:
    raise ValueError("Prompt plus output budget exceeds the recorded context ceiling")

ANSWER = generate_reply(MODEL, TOKENIZER, CUSTOM_PROMPT, max_new_tokens=MAX_NEW_TOKENS, decoding={"do_sample": False})
if not ANSWER:
    raise RuntimeError("Empty response")
RESULT = {"inputId": "prompt-1", "prompt": CUSTOM_PROMPT, "promptTokens": PROMPT_TOKENS, "output": ANSWER, "modelId": ENTRY.model_id, "modelRevision": ENTRY.revision, "decoding": {"doSample": False, "maxNewTokens": MAX_NEW_TOKENS}}
OUTPUT_JSONL = Path("/content/artifact_inference_predictions.jsonl")
OUTPUT_JSONL.write_text(json.dumps(RESULT, ensure_ascii=False) + "\n", encoding="utf-8")
OUTPUT_CSV = Path("/content/artifact_inference_predictions.csv")
pd.DataFrame([RESULT]).to_csv(OUTPUT_CSV, index=False)
OUTPUT_PROVENANCE = Path("/content/artifact_inference_provenance.json")
OUTPUT_PROVENANCE.write_text(json.dumps({
    "profile": "ARTIFACT-INFERENCE", "notebookSpecVersion": "1.0",
    "artifactSha256": sha256_file(ARCHIVE), "artifactFormat": MANIFEST["format"],
    "artifactFormatVersion": MANIFEST["formatVersion"], "baseModel": ENTRY.model_id,
    "baseModelRevision": ENTRY.revision, "pipelineRuntimeRevision": PIPELINE_RUNTIME_REVISION,
    "finetunerRuntimeRevision": FINETUNER_RUNTIME_REVISION,
    "sourceCompatibility": SOURCE_COMPATIBILITY,
    "artifactRuntimeRevisions": RECORDED_RUNTIME_REVISIONS or None,
    "runtime": RUNTIME, "adapterActivity": ACTIVITY, "decoding": RESULT["decoding"],
}, indent=2), encoding="utf-8")
display(pd.DataFrame([RESULT])[["inputId", "prompt", "output"]])
print("wrote", OUTPUT_JSONL, OUTPUT_CSV, OUTPUT_PROVENANCE)


## Interpretation, troubleshooting, and next experiments
A successful run proves archive/manifest validation, model/revision and runtime compatibility, production reconstruction, active adapter deltas, new-input generation, and machine-readable export. It does **not** prove sender authenticity or task quality.

Common failures: fix `GITHUB_TOKEN` access rather than embedding credentials; reject manifest/hash/model mismatches rather than weakening checks; recreate the exact package lock on compatibility mismatch; use suitable GPU hardware for OOM; shorten prompts that exceed the recorded context ceiling.

Next experiments: evaluate multiple domain prompts with a task-specific rubric/human review; compare greedy verification with explicitly recorded stochastic decoding; test another user-facing registry model; verify a whole-ZIP digest obtained via a trusted channel.

Release-grade status still requires the clean-runtime record in [RELEASE_VERIFICATION.md](RELEASE_VERIFICATION.md): clean E2E at the exact candidate head, then a **separate clean ARTIFACT-INFERENCE run** using its downloaded ZIP as external input. Static CI is not REL1/REL5 execution evidence.
